# Reconstructing the `metabolism_redux` FBA problem for a single timestep

This notebook captures the **exact inputs** to `NetworkFlowModel.solve()` at the timestep where
GLOP fails, then rebuilds the CVXPY problem in isolation so we can poke at it (try other solvers,
inspect coefficient magnitudes, toggle the kinetics term, etc.) **without touching the production
code path**.

Workflow:

1. **Part 1 - Capture.** Build `EcoliSim` from `configs/test_PR.json`, monkeypatch
   `NetworkFlowModel.solve` so that on a solver failure it pickles the model arrays + the `solve()`
   kwargs to disk, then run the sim until it fails.
2. **Part 2 - Reconstruct.** Load the pickle and rebuild the LP locally (a faithful copy of the
   current `solve()` body) with knobs for solver / kinetics-on-off / verbosity.
3. **Part 3 - Diagnose.** Inspect the canonical `A`/`c` magnitudes GLOP actually sees, the kinetic
   targets, and compare GLOP vs HiGHS vs Clarabel.

> **Kernel requirement:** run this notebook with the project virtualenv (`.venv/bin/python`, the
> same interpreter `uvenv` uses). Otherwise the `ecoli` imports and `sim_data` won't resolve.

## Part 1 - Capture the failing timestep

We snapshot only the picklable arrays `solve()` actually reads off the model (`S_orig`, `S_exch`,
`exchange_masses`, `gam`, the index arrays, `active_constraints_mask`, `exchanges`) plus the full
kwargs dict. The monkeypatch keeps the snapshot of the **most recent** call and, on a
`SolverError`, dumps that snapshot to `failing_timestep.pkl` and stops the sim.

Re-running is cheap: if `failing_timestep.pkl` already exists you can skip straight to Part 2.

In [1]:
import os
import pickle
from pathlib import Path

import numpy as np
import cvxpy as cp

# Resolve repo root from this notebook's location: .../notebooks/Heena notebooks/Metabolism_New Genes
NB_DIR = Path.cwd()
REPO_ROOT = NB_DIR
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "configs").is_dir():
    REPO_ROOT = REPO_ROOT.parent
print("Repo root:", REPO_ROOT)

CONFIG_PATH = str(REPO_ROOT / "configs" / "test_PR.json")
DUMP_PATH = NB_DIR / "failing_timestep.pkl"
print("Config:", CONFIG_PATH)
print("Dump path:", DUMP_PATH)

# Make sure we can import the ecoli package
import sys

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


Repo root: /Users/heenasaqib/dev/vEcoli
Config: /Users/heenasaqib/dev/vEcoli/configs/test_PR.json
Dump path: /Users/heenasaqib/dev/vEcoli/notebooks/Heena notebooks/Metabolism_New Genes/failing_timestep.pkl


In [2]:
# Attributes of NetworkFlowModel that solve() reads. All picklable (numpy / scipy.sparse / scalars).
_MODEL_ATTRS = [
    "n_orig_rxns",
    "n_exch_rxns",
    "kinetic_rxn_idx",
    "S_orig",
    "S_exch",
    "exchange_masses",
    "gam",
    "intermediates_idx",
    "homeostatic_idx",
    "maintenance_idx",
    "secretion_idx",
    "exchanges",
    "active_constraints_mask",
]


def extract_model_state(model):
    """Pull just the arrays/scalars solve() needs, so we can reconstruct without the full object."""
    state = {}
    for attr in _MODEL_ATTRS:
        state[attr] = getattr(model, attr, None)
    return state


def install_capture():
    """Monkeypatch NetworkFlowModel.solve to snapshot inputs and dump on failure.

    Returns a dict that will hold the last snapshot (also persisted to DUMP_PATH on failure).
    """
    from ecoli.processes.metabolism_redux import NetworkFlowModel

    if getattr(NetworkFlowModel.solve, "_is_capture_wrapper", False):
        orig_solve = NetworkFlowModel.solve._orig_solve
    else:
        orig_solve = NetworkFlowModel.solve

    store = {"call_idx": 0, "last": None, "failed": None}

    def patched_solve(self, *args, **kwargs):
        # solve() in next_update is always called with keyword args
        snapshot = {
            "call_idx": store["call_idx"],
            "model_state": extract_model_state(self),
            "args": args,
            "kwargs": kwargs,
        }
        store["last"] = snapshot
        store["call_idx"] += 1
        try:
            return orig_solve(self, *args, **kwargs)
        except Exception as e:  # noqa: BLE001 - we want to capture any solver failure
            snapshot["error"] = repr(e)
            store["failed"] = snapshot
            with open(DUMP_PATH, "wb") as fh:
                pickle.dump(snapshot, fh)
            print(f"\n[capture] solve() failed on call #{snapshot['call_idx']}: {e!r}")
            print(f"[capture] snapshot written to {DUMP_PATH}")
            raise

    patched_solve._is_capture_wrapper = True
    patched_solve._orig_solve = orig_solve
    NetworkFlowModel.solve = patched_solve
    return store


def uninstall_capture():
    from ecoli.processes.metabolism_redux import NetworkFlowModel

    if getattr(NetworkFlowModel.solve, "_is_capture_wrapper", False):
        NetworkFlowModel.solve = NetworkFlowModel.solve._orig_solve


print("Capture helpers defined.")

Capture helpers defined.


In [3]:
# Run the sim until the solver fails. This takes ~1 minute and stops at the failing timestep.
# Skip this cell if failing_timestep.pkl already exists and you just want to reconstruct.

RUN_CAPTURE = not DUMP_PATH.exists()

if RUN_CAPTURE:
    from ecoli.experiments.ecoli_master_sim import EcoliSim

    store = install_capture()
    sim = EcoliSim.from_file(CONFIG_PATH)
    sim.build_ecoli()
    try:
        sim.run()
        print("Sim finished WITHOUT a solver failure (no snapshot captured this run).")
    except Exception as e:  # noqa: BLE001
        print(f"\nSim stopped due to: {type(e).__name__}: {e}")
    finally:
        uninstall_capture()
else:
    print(f"{DUMP_PATH} already exists - skipping capture. Delete it to re-capture.")


Simulation ID: test_PR_branch_eff_bad_rxn
Created: 06/09/2026 at 14:04:08


/Users/heenasaqib/dev/vEcoli/ecoli/processes/tf_unbinding.py:103: RuntimeWarning: divide by zero encountered in matmul
  mass_diffs = bound_TF @ -self.active_tf_masses
/Users/heenasaqib/dev/vEcoli/ecoli/processes/tf_unbinding.py:103: RuntimeWarning: overflow encountered in matmul
  mass_diffs = bound_TF @ -self.active_tf_masses
/Users/heenasaqib/dev/vEcoli/ecoli/processes/tf_unbinding.py:103: RuntimeWarning: invalid value encountered in matmul
  mass_diffs = bound_TF @ -self.active_tf_masses
/Users/heenasaqib/dev/vEcoli/reconstruction/ecoli/dataclasses/process/metabolism.py:1883: RuntimeWarning: divide by zero encountered in matmul
  counts_per_aa_fwd = enzyme_counts @ self.enzyme_to_amino_acid_fwd
/Users/heenasaqib/dev/vEcoli/reconstruction/ecoli/dataclasses/process/metabolism.py:1883: RuntimeWarning: overflow encountered in matmul
  counts_per_aa_fwd = enzyme_counts @ self.enzyme_to_amino_acid_fwd
/Users/heenasaqib/dev/vEcoli/reconstruction/ecoli/dataclasses/process/metabolism.py:188

/Users/heenasaqib/dev/vEcoli/reconstruction/ecoli/dataclasses/process/metabolism.py:1883: RuntimeWarning: divide by zero encountered in matmul
  counts_per_aa_fwd = enzyme_counts @ self.enzyme_to_amino_acid_fwd
/Users/heenasaqib/dev/vEcoli/reconstruction/ecoli/dataclasses/process/metabolism.py:1883: RuntimeWarning: overflow encountered in matmul
  counts_per_aa_fwd = enzyme_counts @ self.enzyme_to_amino_acid_fwd
/Users/heenasaqib/dev/vEcoli/reconstruction/ecoli/dataclasses/process/metabolism.py:1883: RuntimeWarning: invalid value encountered in matmul
  counts_per_aa_fwd = enzyme_counts @ self.enzyme_to_amino_acid_fwd
/Users/heenasaqib/dev/vEcoli/reconstruction/ecoli/dataclasses/process/metabolism.py:1884: RuntimeWarning: divide by zero encountered in matmul
  counts_per_aa_rev = enzyme_counts @ self.enzyme_to_amino_acid_rev
/Users/heenasaqib/dev/vEcoli/reconstruction/ecoli/dataclasses/process/metabolism.py:1884: RuntimeWarning: overflow encountered in matmul
  counts_per_aa_rev = enzy

(CVXPY) Jun 09 02:04:22 PM: Your problem has 10401 variables, 26480 constraints, and 0 parameters.
(CVXPY) Jun 09 02:04:22 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jun 09 02:04:22 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jun 09 02:04:22 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jun 09 02:04:22 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Jun 09 02:04:22 PM: Using cached ASA map, for faster compilation (bypassing reduction chain).


Lets see what is going on-----------------------------------| 10728.0/10800.0 simulated seconds remaining    
                                     CVXPY                                     
                                     v1.8.1                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) Jun 09 02:04:22 PM: Finished problem compilation (took 5.135e-02 seconds).
(CVXPY) Jun 09 02:04:22 PM: Invoking solver GLOP  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------

Initial problem: 28484 rows, 11403 columns, 65561 entries with magnitude in [2.340372e-05, 3.894685e+09]
Objective stats: 10552 non-zeros, range [1.000000e-07, 1.000000e+00]
Bounds stats: 11334 non-zeros, range [-4.143814e+03, 4.143814e+03]
Parameters: use_preprocessing: true log_search_progress: true

Starting presolve...
SingletonPreprocessor                        : 7323(-21161) rows, 11249(-154) columns, 44246(-21315) entries. (0.001434s)
ForcingAndImpliedFreeConstraintPreprocessor  : 6910(-21574) rows, 10374(-1029) columns, 38586(-26975) entries. (0.000443s)
ImpliedFreePreprocessor                      : 6910(-21574) rows, 10374(-1029) columns, 38586(-26975) entries. (0.000562s)
UnconstrainedVariablePreprocessor            : 6910(-21574) rows, 

## Part 2 - Reconstruct the LP locally

`build_problem()` below is a faithful, self-contained copy of the **current** `solve()` body (the
original in-range/out-of-range two-variable split). It builds the CVXPY `Problem` from the captured
`model_state` + `kwargs`, with one knob:

- `include_kinetics`: drop the kinetics objective term + its box constraints (to confirm the
  observation that the sim is stable without it).

It returns the problem plus the variable handles so we can inspect everything.

> Note: `failing_timestep.pkl` only holds the per-timestep inputs (`model_state` + `kwargs`); it is
> independent of the formulation. To capture a timestep that fails under the *current* formulation,
> delete the pickle and re-run Part 1.

In [4]:
# Load the captured snapshot
with open(DUMP_PATH, "rb") as fh:
    snap = pickle.load(fh)

ms = snap["model_state"]
kw = snap["kwargs"]

print("Captured call index:", snap.get("call_idx"))
print("Original error:      ", snap.get("error"))
print("solve() kwargs:      ", list(kw.keys()))
print()
print("n_orig_rxns:", ms["n_orig_rxns"], " n_exch_rxns:", ms["n_exch_rxns"])
print("S_orig shape:", ms["S_orig"].shape, " S_exch shape:", ms["S_exch"].shape)
print("n kinetic rxns:", None if ms["kinetic_rxn_idx"] is None else len(ms["kinetic_rxn_idx"]))
print("objective_weights:", kw.get("objective_weights"))

Captured call index: 72
Original error:       SolverError("Solver 'GLOP' failed. Try another solver, or solve with verbose=True for more information.")
solve() kwargs:       ['homeostatic_concs', 'homeostatic_dm_targets', 'ngam_target', 'kinetic_targets', 'binary_kinetic_idx', 'objective_weights', 'aa_uptake_package']

n_orig_rxns: 9463  n_exch_rxns: 106
S_orig shape: (6149, 9463)  S_exch shape: (6149, 106)
n kinetic rxns: 416
objective_weights: {'secretion': 0.001, 'efficiency': 1e-05, 'kinetics': 1e-07, 'kinetics_in_range': 0.01}


In [32]:
from typing import cast
import numpy.typing as npt
import time


def build_problem(ms, kwargs, include_kinetics=True):
    """Faithful copy of NetworkFlowModel.solve()'s problem construction
    (original in-range/out-of-range two-variable split).

    include_kinetics : drop the kinetics objective term + its box constraints if False.
    """
    homeostatic_concs = np.array(kwargs["homeostatic_concs"], dtype=float)
    homeostatic_dm_targets = np.array(kwargs["homeostatic_dm_targets"], dtype=float)
    ngam_target = kwargs.get("ngam_target", 0)
    kinetic_targets = kwargs.get("kinetic_targets", None)
    binary_kinetic_idx = kwargs.get("binary_kinetic_idx", None)
    objective_weights = dict(kwargs["objective_weights"])
    aa_uptake_package = kwargs.get("aa_uptake_package", None)
    upper_flux_bound = kwargs.get("upper_flux_bound", 100)

    S_orig = ms["S_orig"]
    S_exch = ms["S_exch"]
    n_orig_rxns = ms["n_orig_rxns"]
    n_exch_rxns = ms["n_exch_rxns"]
    kinetic_rxn_idx = ms["kinetic_rxn_idx"]
    exchange_masses = ms["exchange_masses"]
    gam = ms["gam"]
    intermediates_idx = ms["intermediates_idx"]
    homeostatic_idx = ms["homeostatic_idx"]
    maintenance_idx = ms["maintenance_idx"]
    secretion_idx = ms["secretion_idx"]
    exchanges = ms["exchanges"]
    active_constraints_mask = ms["active_constraints_mask"]

    n_kinetic = len(kinetic_rxn_idx) if kinetic_rxn_idx is not None else 0
    v_diff_in_range = cp.Variable(n_kinetic)
    v_diff_outside_range = cp.Variable(n_kinetic)
    v = cp.Variable(n_orig_rxns)
    e = cp.Variable(n_exch_rxns)
    dm = S_orig @ v + S_exch @ e
    exch = S_exch @ e
    total_maintenance = ngam_target + gam * e @ exchange_masses

    constr = []
    constr.append(dm[intermediates_idx] == 0)
    constr.append(
        v[kinetic_rxn_idx]
        == kinetic_targets[:, 1] + v_diff_in_range + v_diff_outside_range
    )

    if maintenance_idx is not None:
        constr.append(v[maintenance_idx] == total_maintenance)
        constr.append(v[maintenance_idx] >= ngam_target)
    if binary_kinetic_idx is not None and len(binary_kinetic_idx) > 0:
        constr.append(v[binary_kinetic_idx] == 0)
    constr.extend([v >= 0, v <= upper_flux_bound, e >= 0, e <= upper_flux_bound])

    if aa_uptake_package:
        levels, molecules, force = aa_uptake_package
        for level, mol in zip(levels, molecules):
            exch_idx = exchanges.index(mol + " exchange")
            constr.append(e[exch_idx] == level)

    homeostatic_target_concs = homeostatic_concs + homeostatic_dm_targets
    homeostatic_target_concs[homeostatic_target_concs == 0] = 1

    loss = 0
    loss += cp.norm1(
        (dm[homeostatic_idx] - homeostatic_dm_targets) / homeostatic_concs
    )
    if "secretion" in objective_weights:
        loss += objective_weights["secretion"] * cp.sum(
            e[secretion_idx] @ -exchange_masses[secretion_idx]
        )
    if include_kinetics and "kinetics" in objective_weights:
        kinetic_targets = cast(npt.NDArray[np.float64], kinetic_targets)
        # Fix divide by zero
        nonzero_kinetic_targets = kinetic_targets[:, 1].copy()
        nonzero_kinetic_targets[nonzero_kinetic_targets == 0] = 1
        # Lower and upper limit for flux diff
        lower_flux_diff = kinetic_targets[:, 0] - kinetic_targets[:, 1]
        upper_flux_diff = kinetic_targets[:, 2] - kinetic_targets[:, 1]
        constr.extend(
            [
                v_diff_in_range >= lower_flux_diff,
                v_diff_in_range <= upper_flux_diff,
            ]
        )
        # Heavily weight fluxes outside limits
        loss += (
            # objective_weights["kinetics"] *
            cp.norm1(
                # v_diff_outside_range
                (v_diff_outside_range / nonzero_kinetic_targets)[active_constraints_mask]
            ))
        # Lightly weight fluxes in expected range
        loss += (
                # objective_weights["kinetics"] *
                objective_weights["kinetics_in_range"] *
                cp.norm1(
                    # v_diff_in_range
                    (v_diff_in_range / nonzero_kinetic_targets)[active_constraints_mask]
                )
        )

    if "efficiency" in objective_weights:
        # Efficiency objective to minimize total flux (proxy for enzyme usage)
        loss += objective_weights["efficiency"] * cp.sum(v)

    p = cp.Problem(cp.Minimize(loss), constr)
    handles = dict(
        v=v,
        e=e,
        dm=dm,
        exch=exch,
        total_maintenance=total_maintenance,
        v_diff_in_range=v_diff_in_range,
        v_diff_outside_range=v_diff_outside_range,
    )
    return p, handles


def solve_and_report(p, solver=cp.GLOP, verbose=False, **solve_kw):
    t0 = time.time()
    status, obj, err = None, None, None
    try:
        p.solve(solver=solver, verbose=verbose, **solve_kw)
        status, obj = p.status, p.value
    except Exception as e:  # noqa: BLE001
        status, err = "EXCEPTION", repr(e)
    dt = time.time() - t0
    label = f"{str(solver):>9}"
    print(f"solver={label}  status={str(status):>22}  obj={obj}  time={dt:.3f}s")
    if err:
        print("   error:", err)
    return status, obj, err


print("build_problem() and solve_and_report() defined.")

build_problem() and solve_and_report() defined.


In [33]:
from ortools.glop import parameters_pb2

params = parameters_pb2.GlopParameters()
dir(params)

['ByteSize',
 'Clear',
 'ClearExtension',
 'ClearField',
 'CopyFrom',
 'DESCRIPTOR',
 'DiscardUnknownFields',
 'FindInitializationErrors',
 'FromString',
 'HasExtension',
 'HasField',
 'IsInitialized',
 'ListFields',
 'MergeFrom',
 'MergeFromString',
 'ParseFromString',
 'SerializePartialToString',
 'SerializeToString',
 'SetInParent',
 'UnknownFields',
 'WhichOneof',
 '_CheckCalledFromGeneratedFile',
 '_ListFieldsItemKey',
 '_SetListener',
 '__class__',
 '__contains__',
 '__deepcopy__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '__unicode__',
 'allow_simplex_algorithm_change',
 'basis_refactorization_period',
 'change_status_to_imprecise',
 'cost_scaling',


In [34]:
params.cost_scaling
# params.solution_feasibility_tolerance
params.primal_feasibility_tolerance

1e-08

In [35]:
# params.cost_scaling = 3
params.solution_feasibility_tolerance = 1E-6
# params.relative_max_cost_perturbation = 1E-5
p, H = build_problem(ms, kw, include_kinetics=True)
print(f"variables={sum(var.size for var in p.variables())}  constraints={len(p.constraints)}")
solve_and_report(p, solver=cp.GLOP, verbose=True, parameters_proto=params)

(CVXPY) Jun 09 02:06:57 PM: Your problem has 10401 variables, 26480 constraints, and 0 parameters.
(CVXPY) Jun 09 02:06:57 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jun 09 02:06:57 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jun 09 02:06:57 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jun 09 02:06:57 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Jun 09 02:06:57 PM: Compiling problem (target solver=GLOP).
(CVXPY) Jun 09 02:06:57 PM: Reduction chain: Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> GLOP
(CVXPY) Jun 09 02:06:57 PM: Applying reduction Dcp2Cone
(CVXPY) Jun 09 02:06:57 PM: Applying reduction CvxAttr2Constr
(CVXPY) Jun 09 02:06:57 PM: Applying reduction ConeMatrixStuffing
(CVXPY) Jun 09 02:06:57 PM: Applying reduction GLOP
(CVXPY) Jun 09 02:06:57 PM: Finished problem compilation (t

variables=10401  constraints=11
                                     CVXPY                                     
                                     v1.8.1                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------

Initial problem: 28484 rows, 11403 columns, 65561 entries with magnitude in [2.340372e-05, 3.894685e+09]
Objective stats: 10552 non-zeros, range [1.008000e-06, 1.000000e+00]
Bounds stats: 11334 non-zeros, range [-4.143814e+03, 4.143814e+03]
Parameters: solution_feasibility_tolerance: 1e-06 use_preprocessing: true log_search_p

('EXCEPTION',
 None,
 'SolverError("Solver \'GLOP\' failed. Try another solver, or solve with verbose=True for more information.")')

In [39]:
kt = kw["kinetic_targets"]
bad = np.where(np.isnan(kt).any(axis=1))[0]
print(bad)
kt[bad]

[75]


array([[0.00211056,        nan, 0.00493733]])

In [48]:
# A/B on the SAME instance: is the kinetics term what stresses GLOP?
print("== kinetics OFF ==")
p_off, _ = build_problem(ms, kw, include_kinetics=False)
solve_and_report(p_off, solver=cp.GLOP)

print("\n== kinetics ON ==")
p_on, _ = build_problem(ms, kw, include_kinetics=True)
solve_and_report(p_on, solver=cp.GLOP)

== kinetics OFF ==
solver=     GLOP  status=               optimal  obj=3.1985370264762705  time=0.102s

== kinetics ON ==
solver=     GLOP  status=             EXCEPTION  obj=None  time=0.108s
   error: SolverError("Solver 'GLOP' failed. Try another solver, or solve with verbose=True for more information.")


('EXCEPTION',
 None,
 'SolverError("Solver \'GLOP\' failed. Try another solver, or solve with verbose=True for more information.")')

In [49]:
# Does a different solver handle the same (kinetics-on) instance cleanly?
for solver in [cp.GLOP, cp.HIGHS, cp.CLARABEL, cp.SCIPY]:
    if solver not in cp.installed_solvers():
        print(f"{solver}: not installed, skipping")
        continue
    p_try, _ = build_problem(ms, kw, include_kinetics=True)
    solve_and_report(p_try, solver=solver)

solver=     GLOP  status=             EXCEPTION  obj=None  time=0.120s
   error: SolverError("Solver 'GLOP' failed. Try another solver, or solve with verbose=True for more information.")
solver=    HIGHS  status=               optimal  obj=3.1987156977947726  time=0.100s
solver= CLARABEL  status=               optimal  obj=3.1987350686360587  time=0.711s
solver=    SCIPY  status=             EXCEPTION  obj=None  time=0.168s
   error: SolverError("Solver 'SCIPY' failed. Try another solver, or solve with verbose=True for more information.")


## Part 3 - Diagnose

Two things to look at:

1. **Canonical coefficient magnitudes** GLOP actually receives (`get_problem_data`). This mirrors the
   `entries with magnitude in [...]` / `Objective stats` lines from the GLOP verbose log and exposes
   the conditioning of the *kinetics-on* instance.
2. **Kinetic target pathologies** - tiny/zero/negative centers and inverted bands feed the `1/target`
   normalizer and are prime suspects for blowing up the objective coefficients.

In [25]:
import scipy.sparse as sp


def mag_stats(name, M):
    if sp.issparse(M):
        d = np.abs(M.data)
    else:
        d = np.abs(np.asarray(M, dtype=float).ravel())
    d = d[d > 0]
    if d.size == 0:
        print(f"{name:>6}: all zero")
        return
    print(
        f"{name:>6}: nnz={d.size:>7}  |min|={d.min():.3e}  |max|={d.max():.3e}  "
        f"spread={d.max() / d.min():.3e}"
    )


def report_conditioning(p, label):
    print(f"--- {label} ---")
    data, _, _ = p.get_problem_data(cp.GLOP)
    for k, val in data.items():
        if sp.issparse(val) or (hasattr(val, "shape") and getattr(val, "ndim", 0) >= 1):
            mag_stats(k, val)
    print()


report_conditioning(build_problem(ms, kw, include_kinetics=True)[0], "kinetics ON")
report_conditioning(build_problem(ms, kw, include_kinetics=False)[0], "kinetics OFF")

--- kinetics ON (deadband) ---

--- kinetics OFF ---



In [26]:
# Kinetic target pathologies (these feed the 1/target normalizer in the objective)
kt = np.asarray(kw["kinetic_targets"], dtype=float)
lower, center, upper = kt[:, 0], kt[:, 1], kt[:, 2]
mask = np.asarray(ms["active_constraints_mask"])

print("kinetic_targets shape:", kt.shape, " active constraints:", int(mask.sum()))
print("center (col 1) range: [%.3e, %.3e]" % (center.min(), center.max()))
print()
print("centers == 0:                 ", int((center == 0).sum()))
print("centers  < 0:                 ", int((center < 0).sum()))
print("inverted bands (lower>upper): ", int((lower > upper).sum()))
print("center outside [lower,upper]: ", int(((center < lower) | (center > upper)).sum()))

active_center = center[mask]
nz = np.abs(active_center[active_center != 0])
if nz.size:
    print()
    print("among ACTIVE constraints:")
    print("  min |nonzero center|:", "%.3e" % nz.min())
    print("  max |nonzero center|:", "%.3e" % nz.max())
    print("  implied 1/min center:", "%.3e" % (1.0 / nz.min()))
    # Worst offenders
    order = np.argsort(nz)
    print("  10 smallest |center| (active):", np.round(np.sort(nz)[:10], 12))

kinetic_targets shape: (416, 3)  active constraints: 415
center (col 1) range: [0.000e+00, 5.138e+01]

centers == 0:                  8
centers  < 0:                  0
inverted bands (lower>upper):  0
center outside [lower,upper]:  0

among ACTIVE constraints:
  min |nonzero center|: 1.068e-08
  max |nonzero center|: 5.138e+01
  implied 1/min center: 9.361e+07
  10 smallest |center| (active): [1.06820e-08 1.83520e-08 9.20550e-08 1.22632e-07 1.46582e-07 2.48091e-07
 3.33054e-07 3.33351e-07 4.35667e-07 4.74397e-07]


### How to use this notebook

- **First run:** execute Part 1 to capture `failing_timestep.pkl`. Subsequent runs auto-skip capture.
- **Iterate on the formulation:** edit `build_problem()` (Part 2) freely - it's a standalone copy, so
  nothing here touches the production `metabolism_redux.py`. Once a variant solves cleanly here, port
  it back into `solve()`.
- **Knobs to sweep:** `include_kinetics`, `solver`, and the `objective_weights` inside `kw` (e.g. try
  bumping `kinetics`/`kinetics_in_range` up by a few orders, or floor `homeostatic_target_concs`, to
  see the effect on conditioning and solve status).

`snap`, `ms` (model arrays), and `kw` (solve kwargs) are all in scope for ad-hoc inspection.